In [ ]:
%pip install -q polars plotly gdown python-dotenv kailash-ml kailash-pact kailash-nexus kailash kailash-dataflow


# ════════════════════════════════════════════════════════════════════════
# ASCENT10 — Exercise 7: Enterprise ML Governance at Scale
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build large-scale ML governance infrastructure -- model
#   governance registry via DataFlow, policy engine, automated retirement,
#   multi-tenancy isolation, and governance incident simulation.
#
# TASKS:
#   1. Design model governance registry using DataFlow (@db.model + db.express)
#   2. Implement governance policy engine with structured verdicts
#   3. Model retirement automation: DriftMonitor -> PACT clearance downgrade
#   4. Multi-tenancy isolation: shared InferenceServer, isolated governance
#   5. Governance incident simulation: bias -> alert -> audit -> remediation
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import hashlib
import json
import os
import tempfile
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from dataflow import DataFlow
from kailash_ml.engines.drift_monitor import DriftMonitor
from kailash.trust import ConfidentialityLevel, TrustPosture
from pact import GovernanceEngine, PactGovernedAgent, compile_org, load_org_yaml
from pact.governance import RoleClearance, RoleEnvelope
from kailash.trust.pact.config import (
    ConstraintEnvelopeConfig,
    OperationalConstraintConfig,
    TemporalConstraintConfig,
    DataAccessConstraintConfig,
    CommunicationConstraintConfig,
)
from nexus import Nexus

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



## TASK 1: Model governance registry using DataFlow


In [ ]:
print("=== Model Governance Registry ===\n")

# TODO: Instantiate DataFlow against an in-memory SQLite database.
db = ____


@db.model
class GovernedModel:
    """Registry entry for a governed ML model."""

    # TODO: Declare fields: id (int PK), model_id, model_name, version,
    # TODO: training_data_hash, adapter_history, clearance_requirement,
    # TODO: drift_psi_latest=0.0, drift_check_count=0, compliance_status="PENDING",
    # TODO: retirement_status="ACTIVE", registered_at="", last_audit_at="".
    ____


@db.model
class GovernanceEvent:
    """Audit log entry for governance actions."""

    # TODO: id (int PK), event_type, model_id, actor, details (json), timestamp.
    ____


async def setup_registry():
    """Initialise DataFlow and create governance tables."""
    # TODO: await db.initialize().
    # TODO: Build a list of 3 sample models (fraud_detector_v1, fraud_detector_v2,
    # TODO: icu_predictor_v1) with hash values, clearance requirements, statuses.
    # TODO: For each, await db.express.create("GovernedModel", model_data).
    # TODO: List all rows and print a header + summary table.
    ____


dataflow, registered_models  = await setup_registry()


## TASK 2: Governance policy engine with structured verdicts


In [ ]:
print(f"\n{'=' * 70}")
print(f"=== Governance Policy Engine ===")
print(f"{'=' * 70}\n")


# TODO: Define GOVERNANCE_POLICIES as a list of dicts with keys policy_id,
# TODO: name, description, check (lambda m -> bool), severity (BLOCKING/WARNING).
# TODO: Cover compliance status, drift threshold, training data provenance,
# TODO: retirement status, audit recency.
GOVERNANCE_POLICIES = ____


def apply_policies(model_metadata: dict) -> dict:
    """Apply all governance policies and produce a structured verdict."""
    # TODO: Walk every policy. Append PASS/FAIL records.
    # TODO: For BLOCKING fails set overall=BLOCKED + record violation.
    # TODO: For WARNING fails set APPROVED_WITH_WARNINGS unless already BLOCKED.
    ____


# TODO: For each registered model run apply_policies and print verdict block.
____



## TASK 3: Model retirement automation — DriftMonitor -> clearance downgrade


In [ ]:
print(f"{'=' * 70}")
print(f"=== Automated Model Retirement ===")
print(f"{'=' * 70}\n")

rng = np.random.default_rng(42)
reference_data = rng.normal(0, 1, (1000, 10))
feature_names = [f"v{i}" for i in range(10)]

# TODO: Instantiate DriftMonitor with reference_data, feature_names, psi_threshold=0.25.
monitor = ____

# TODO: Define a YAML org with ops_lead and model_agent (agent: true);
# TODO: clearances at secret/confidential on relevant compartments.
retire_org_yaml = ____

# TODO: Write to temp, load, compile, build GovernanceEngine.
retire_file = ____
____
retire_loaded = ____
retire_engine = ____
retire_compiled = ____

retire_roles = ____
____

level_map = ____

# TODO: Grant clearances for ops_lead and model_agent.
____


async def simulate_retirement_automation():
    """Simulate consecutive drift -> clearance downgrade + retirement event."""
    # TODO: Run check_drift on stable, mildly drifted, then heavily drifted batches.
    # TODO: If both later checks have_drift, downgrade model_agent to RESTRICTED
    # TODO: clearance, await db.express.update("GovernedModel", id, {...}),
    # TODO: log a GovernanceEvent of event_type=RETIREMENT, fetch the updated
    # TODO: model, and re-apply policies. Print every step.
    ____


await simulate_retirement_automation()
retire_file.unlink()



## TASK 4: Multi-tenancy isolation — shared infrastructure, isolated governance


In [ ]:
print(f"\n{'=' * 70}")
print(f"=== Multi-Tenancy Isolation ===")
print(f"{'=' * 70}\n")

# TODO: Define two isolated YAML orgs (bank_alpha + bank_beta) each with
# TODO: a single agent role and tenant-specific compartments.
org_a_yaml = ____

org_b_yaml = ____

# TODO: For each, write to temp file, load_org_yaml, compile_org, build engine.
file_a = ____
____
file_b = ____
____

loaded_a = ____
loaded_b = ____

engine_a = ____
engine_b = ____

compiled_a = ____
compiled_b = ____

roles_a = ____
____

roles_b = ____
____

# TODO: Grant CONFIDENTIAL clearances to each org's agent on its own compartments.
____

____

# TODO: Build KnowledgeItems (alpha_fraud_model, beta_fraud_model) and
# TODO: confirm each agent can only access its own org's resource.
from pact.governance import KnowledgeItem

alpha_model = ____

beta_model = ____

print(f"Bank Alpha agent accessing Alpha model:")
result_a = ____
print(f"  {'ALLOW' if result_a.allowed else 'DENY'}: {result_a.reason}")

print(f"\nBank Beta agent accessing Beta model:")
result_b = ____
print(f"  {'ALLOW' if result_b.allowed else 'DENY'}: {result_b.reason}")

print(f"\nIsolation guarantee:")
print(f"  Separate GovernanceEngines per organisation")
print(f"  Bank Alpha roles do not exist in Bank Beta's engine (and vice versa)")
print(f"  Shared InferenceServer routes requests through org-specific governance")
print(f"  Audit trails are per-org (no cross-tenant log leakage)")

# TODO: Verify and report each engine's audit integrity, then unlink temp files.
integrity_a = ____
integrity_b = ____
print(f"\n  Bank Alpha audit integrity: {'VALID' if integrity_a else 'BROKEN'}")
print(f"  Bank Beta audit integrity: {'VALID' if integrity_b else 'BROKEN'}")

____
____



## TASK 5: Governance incident simulation


In [ ]:
print(f"\n{'=' * 70}")
print(f"=== Governance Incident Simulation ===")
print(f"{'=' * 70}\n")

loader = ASCENTDataLoader()
fraud_data = loader.load("ascent04", "credit_card_fraud.parquet")

# TODO: Build feature matrix + target as in earlier exercises and stratified split.
feature_cols = ____
X = ____
y = ____

X_train, X_test, y_train, y_test = ____

# TODO: Train an incident_model = GradientBoostingClassifier (n_estimators=30, max_depth=4).
incident_model = ____
____

# TODO: Identify high_value (>75th pct) and low_value (<=25th pct) masks
# TODO: on the test set's amount column. Compute positive prediction rates
# TODO: for each segment and the gap.
amount_test = ____
high_value_mask = ____
low_value_mask = ____

y_pred = ____
high_value_fpr = ____
low_value_fpr = ____
fpr_gap = ____

print(f"Incident: Biased fraud prediction rates detected")
print(f"  High-value transaction FPR: {high_value_fpr:.4f}")
print(f"  Low-value transaction FPR: {low_value_fpr:.4f}")
print(f"  FPR gap: {fpr_gap:.4f} ({'ALERT' if fpr_gap > 0.05 else 'OK'})")

if fpr_gap > 0.02:
    # TODO: Print the D/T/R incident response chain (alert -> audit -> root
    # TODO: cause -> remediation -> accountability) using the values above.
    ____

    # TODO: Define an async function log_incident() that writes a
    # TODO: GovernanceEvent row with event_type=POLICY_VIOLATION and a
    # TODO: details JSON containing the FPRs + remediation; asyncio.run it.
    async def log_incident():
        ____

    await log_incident()
# TODO: Close the dataflow connection and remove the temporary db file.
____

db_path = Path("ascent10_governance.db")
if db_path.exists():
    db_path.unlink()

print("\n--- Exercise 7 complete: enterprise ML governance at scale ---")

